# 09 Survival Analysis — Reference Solutions

Complete solutions for the survival analysis exercises on the Songbai Nursing Home Legionella outbreak.

In [ ]:
# Google Colab setup -- 若在本機執行可跳過此 cell
import sys
import os
if 'google.colab' in sys.modules:
    !git clone https://github.com/ancientsky/python4epi.git /content/python4epi 2>/dev/null || true
    os.chdir('/content/python4epi')
    !pip install -q -e .

In [ ]:
import pathlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from lifelines import KaplanMeierFitter, CoxPHFitter
from lifelines.statistics import logrank_test

# -- CJK font setup (avoids Chinese labels showing as boxes) --
# Scan system font directories and explicitly register CJK fonts (more reliable than relying on the cache)
for _font_dir in map(pathlib.Path, ["/usr/share/fonts", "/usr/local/share/fonts"]):
    if _font_dir.exists():
        for _fp in sorted(_font_dir.rglob("*")):
            if _fp.suffix.lower() in {".ttf", ".ttc", ".otf"} and (
                "CJK" in _fp.name or "WenQuanYi" in _fp.name or "wqy" in _fp.name
            ):
                try:
                    fm.fontManager.addfont(str(_fp))
                except Exception:
                    pass

plt.rcParams["font.sans-serif"] = [
    "Noto Sans CJK TC", "Noto Sans CJK SC", "Noto Sans CJK JP",
    "Noto Sans TC", "Microsoft JhengHei",
    "WenQuanYi Zen Hei", "SimHei", "Arial Unicode MS",
    "Heiti TC", "DejaVu Sans",
]
plt.rcParams["axes.unicode_minus"] = False
plt.style.use("ggplot")
plt.rcParams["figure.dpi"] = 150

df = pd.read_csv("data/synthetic/legionella_outbreak.csv")
df["symptom_onset_date"] = pd.to_datetime(df["symptom_onset_date"], errors="coerce")
df["death_date"] = pd.to_datetime(df["death_date"], errors="coerce")
df["infected"] = (df["clinical_severity"] != "not_ill").astype(int)

cases = df[df["infected"] == 1].copy()
cases["event"] = (cases["outcome"] == "dead").astype(int)

investigation_end = cases["symptom_onset_date"].max() + pd.Timedelta(days=14)
cases["end_date"] = cases.apply(
    lambda r: r["death_date"] if r["event"] == 1 else investigation_end, axis=1
)
cases["time_to_event"] = (cases["end_date"] - cases["symptom_onset_date"]).dt.days

## Question 1: Survival Analysis for CHF (Congestive Heart Failure)

In [ ]:
# KM curves: CHF vs No CHF
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("CHF", cases["comorbidity_chf"] == 1),
                     ("No CHF", cases["comorbidity_chf"] == 0)]:
    sub = cases[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("Survival curves: CHF vs No CHF")
ax.set_xlabel("Days since onset")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank test
chf_yes = cases[cases["comorbidity_chf"] == 1]
chf_no = cases[cases["comorbidity_chf"] == 0]

result = logrank_test(
    chf_yes["time_to_event"], chf_no["time_to_event"],
    event_observed_A=chf_yes["event"],
    event_observed_B=chf_no["event"],
)

print(f"Log-rank test statistic = {result.test_statistic:.3f}")
print(f"p-value = {result.p_value:.4f}")

if result.p_value < 0.05:
    print("\n→ p < 0.05: CHF significantly affects survival")
else:
    print("\n→ p ≥ 0.05: CHF's effect on survival is not statistically significant")
    print("→ Possibly because the sample is too small (only 19 deaths), so the test is underpowered")

## Question 2: Survival Comparison by Age Group

In [ ]:
# Age grouping
cases["age_group"] = np.where(cases["age"] >= 75, "≥75", "<75")

fig, ax = plt.subplots(figsize=(8, 5))

for label in ["≥75", "<75"]:
    sub = cases[cases["age_group"] == label]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"Age {label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("Survival curves: Age ≥75 vs <75")
ax.set_xlabel("Days since onset")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank
old = cases[cases["age"] >= 75]
young = cases[cases["age"] < 75]

result_age = logrank_test(
    old["time_to_event"], young["time_to_event"],
    event_observed_A=old["event"],
    event_observed_B=young["event"],
)

print(f"Log-rank test statistic = {result_age.test_statistic:.3f}")
print(f"p-value = {result_age.p_value:.4f}")

if result_age.p_value < 0.05:
    print("\n→ The elderly (≥75) group has significantly worse survival")
else:
    print("\n→ The effect of age grouping on survival is not statistically significant")
    print("→ Nursing home residents are generally older, so the between-group difference may not be large enough")

## Question 3 (Challenge): Hospitalized vs Not Hospitalized + Cox Regression

In [ ]:
# KM curves: hospitalized vs not hospitalized
fig, ax = plt.subplots(figsize=(8, 5))

for label, mask in [("Hospitalized", cases["hospitalized"] == 1),
                     ("Not hospitalized", cases["hospitalized"] == 0)]:
    sub = cases[mask]
    kmf = KaplanMeierFitter()
    kmf.fit(sub["time_to_event"], event_observed=sub["event"],
            label=f"{label} (n={len(sub)}, deaths={sub['event'].sum()})")
    kmf.plot_survival_function(ax=ax)

ax.set_title("Survival curves: Hospitalized vs Not hospitalized")
ax.set_xlabel("Days since onset")
ax.set_ylabel("Survival probability")
ax.set_ylim(0, 1.05)
ax.legend(loc="lower left")
plt.tight_layout()
plt.show()

# Log-rank
hosp_yes = cases[cases["hospitalized"] == 1]
hosp_no = cases[cases["hospitalized"] == 0]

result_hosp = logrank_test(
    hosp_yes["time_to_event"], hosp_no["time_to_event"],
    event_observed_A=hosp_yes["event"],
    event_observed_B=hosp_no["event"],
)

print(f"Log-rank test statistic = {result_hosp.test_statistic:.3f}")
print(f"p-value = {result_hosp.p_value:.4f}")

In [ ]:
# Cox regression
cox_df = cases[[
    "time_to_event", "event", "age", "sex",
    "hospitalized", "comorbidity_copd", "comorbidity_chf",
]].copy()
cox_df["is_male"] = (cox_df["sex"] == "M").astype(int)
cox_df = cox_df.drop(columns=["sex"])

cph = CoxPHFitter()
cph.fit(cox_df, duration_col="time_to_event", event_col="event")

print("=== Cox regression results ===")
summary = cph.summary[["exp(coef)", "exp(coef) lower 95%", "exp(coef) upper 95%", "p"]]
summary.columns = ["HR", "HR_lower", "HR_upper", "p_value"]
print(summary.round(3).to_string())

# HR forest plot
fig, ax = plt.subplots(figsize=(8, 5))
cph.plot(ax=ax)
ax.axvline(x=0, color="gray", linestyle="--", alpha=0.5)
ax.set_title("Cox Regression — HR forest plot")
plt.tight_layout()
plt.show()

print("\n=== Interpretation ===")
hosp_hr = cph.summary.loc["hospitalized", "exp(coef)"]
print(f"hospitalized HR = {hosp_hr:.3f}")
if hosp_hr > 1:
    print("→ Hospitalized patients have HR > 1, seemingly 'increasing' the risk of death")
    print("→ But this does not mean hospitalization is a risk factor!")
    print("→ People are hospitalized because they are severely ill—this is confounding by indication")
    print("→ Hospitalization is a 'marker' of severity, not a 'cause' of death")
else:
    print("→ Hospitalized patients have HR < 1; after adjusting for other factors, hospitalization may have a protective effect")
    print("→ But interpret with caution—confounding by indication still applies")

### Key Interpretation Points

- **CHF**: heart failure may increase the risk of death, but in a small sample it may not reach statistical significance
- **Age**: nursing home residents are generally older, so the between-group difference may be small
- **Hospitalization**: this is a classic case of **confounding by indication** in survival analysis
  - The mortality of hospitalized patients may be higher, but the reason is that "more severely ill people are the ones who get hospitalized"
  - Hospitalization itself is a treatment action that should reduce the risk of death
  - But in observational data, the HR for hospitalization may be > 1 because it is a marker of severity
- **Limitation**: this case has only 19 deaths; putting too many variables in a Cox model easily overfits. The recommendation is at least 10 events per variable, so keeping at most 1-2 variables is more stable